Imports


In [ ]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML, display


Sun, Earth, and Moon: Initial Conditions and Physical Scales

In [ ]:
# ============================================================
# CELL 2 — SUN, EARTH, AND MOON : INITIAL CONDITIONS AND PHYSICAL SCALES 
# ============================================================


# ------------------------------------------------------------
# BODY MASSES
# Set each body's mass independently, in kilograms. The array order is used
# throughout the notebook: masses[0] = Sun, masses[1] = Earth, and masses[2] = Moon.
# ------------------------------------------------------------

masses = np.array([
    1.9884e30,                     # Sun (kg)
    5.9722e24,                     # Earth (kg)
    7.346e22                       # Moon (kg)
], dtype=float)

# ------------------------------------------------------------
# INITIAL POSITIONS
# Each row contains one body's dimensionless (x, y, z) coordinates.
# Physical coordinates are calculated later by multiplying by LENGTH_SCALE.
# ------------------------------------------------------------

initial_positions = np.array([
    [ 0.0, 0.0,  0.0],          # Sun: (x, y, z) * L₀
    [ 1.0, 0.0,  0.0],          # Earth: (x, y, z) * L₀
    [ 1.0025695553, 0.0, 0.0]   # Moon: (x, y, z) * L₀
], dtype=float)


# ------------------------------------------------------------
# INITIAL VELOCITIES
# Each row contains one body's dimensionless (vx, vy, vz) components.
# Physical velocities are calculated later by multiplying by VELOCITY_SCALE.
# ------------------------------------------------------------

initial_velocities = np.array([
    [0, 0, 0],            # Sun: (vx, vy, vz) * v₀
    [0, 1.0, 0],          # Earth: (vx, vy, vz) * v₀
    [0, 1.0343057187, 0]  # Moon: (vx, vy, vz) * v₀
], dtype=float)


# ------------------------------------------------------------
# PHYSICAL CONSTANTS
# ------------------------------------------------------------

G = 6.67430e-11                 # Gravitational constant (m^3 kg^-1 s^-2)


# ------------------------------------------------------------
# CHARACTERISTIC LENGTH SCALE
# One dimensionless position unit represents LENGTH_SCALE meters.
# ------------------------------------------------------------

LENGTH_SCALE = 149597870700         # Characteristic length, L₀ (m) , which is equal to 1 Astronomical Unit (AU)


# ------------------------------------------------------------
# CHARACTERISTIC GRAVITATIONAL SCALES
# The largest body mass is used as the reference mass. These scales convert
# dimensionless positions, velocities, and time into SI units.
# ------------------------------------------------------------

MASS_SCALE = np.max(masses)
TIME_SCALE = np.sqrt(LENGTH_SCALE**3 / (G * MASS_SCALE))
VELOCITY_SCALE = LENGTH_SCALE / TIME_SCALE


# ------------------------------------------------------------
# VALIDATE INPUTS
# Check dimensions and finite, physically valid values before integration.
# ------------------------------------------------------------

if masses.shape != (3,):
    raise ValueError("masses must contain exactly three values.")
if not np.all(np.isfinite(masses)) or np.any(masses <= 0):
    raise ValueError("All body masses must be finite and greater than zero.")
if initial_positions.shape != (3, 3):
    raise ValueError("initial_positions must have shape (3, 3).")
if initial_velocities.shape != (3, 3):
    raise ValueError("initial_velocities must have shape (3, 3).")
if not np.all(np.isfinite(initial_positions)):
    raise ValueError("Initial positions must contain finite values.")
if not np.all(np.isfinite(initial_velocities)):
    raise ValueError("Initial velocities must contain finite values.")
if not np.isfinite(LENGTH_SCALE) or LENGTH_SCALE <= 0:
    raise ValueError("LENGTH_SCALE must be finite and greater than zero.")


print("PHYSICAL SCALES")
print(f"Reference mass: {MASS_SCALE:.6e} kg")
print(f"Length scale (L₀): 1 AU = {LENGTH_SCALE} m")
print(f"Time scale: {TIME_SCALE} s")
print(f"Velocity scale (v₀): {VELOCITY_SCALE} m/s")



Simulation Parameters

In [ ]:
# ============================================================
# CELL 3 — SIMULATION PARAMETERS
# ============================================================


# ------------------------------------------------------------
# SIMULATION DURATION
# simulation_duration is measured in characteristic time units.
# ------------------------------------------------------------

simulation_duration = 250

t_start = 0.0
t_end = simulation_duration * TIME_SCALE


# ------------------------------------------------------------
# VIDEO SETTINGS
# video_duration and fps determine the playback length and frame rate.
# num_frames is the number of samples used in the displayed animation.
# ------------------------------------------------------------

video_duration = 60          # Video duration (seconds)
fps = 60
num_frames = video_duration * fps


# ------------------------------------------------------------
# COLLISION AND EJECTION THRESHOLDS
# A collision is detected when a pair's separation falls below collision_radius.
# Ejection requires a body to be distant and energetically unbound relative to
# the other two bodies.
# ------------------------------------------------------------

collision_radius = 0.001 * LENGTH_SCALE
ejection_radius = 500.0 * LENGTH_SCALE


# ------------------------------------------------------------
# NUMERICAL INTEGRATION SETTINGS
# rtol and atol control the baseline solver accuracy. Smaller values generally
# require more computation. 'tolerances' defines the additional settings used
# in the numerical convergence comparison.
# ------------------------------------------------------------

rtol = 1e-12
atol = 1e-14

tolerances = [
    (1e-8,  1e-10),
    (1e-10,  1e-12),
    (1e-12, 1e-14),
]


print("\nSIMULATION PARAMETERS")
print(f"Physical simulation duration: {t_end} s")
print(f"Video duration: {video_duration} seconds")
print(f"Total video frames: {num_frames}")
print(f"Collision radius: {collision_radius} m")
print(f"Ejection radius: {ejection_radius} m")


Convert Initial Conditions to SI Units

In [ ]:
# ============================================================
# CELL 4 — CONVERT INITIAL CONDITIONS TO SI UNITS
# ============================================================

# Convert the dimensionless starting positions and velocities entered in Cell 3
# into meters and meters per second, respectively, for the numerical solver.
def make_initial_conditions(initial_positions, initial_velocities):
    """Convert dimensionless initial conditions into SI units."""

    positions_si = LENGTH_SCALE * initial_positions
    velocities_si = VELOCITY_SCALE * initial_velocities

    return positions_si, velocities_si


Newtonian Equations of Motion in 3D

In [ ]:
# ============================================================
# CELL 5 — NEWTONIAN EQUATIONS OF MOTION
# ============================================================

# The solver state contains nine position components followed by nine velocity
# components. Return their derivatives: the three velocities followed by the
# gravitational accelerations of the three bodies in x, y, and z.
def derivatives(t, state):
    """Return the first-order 3D equations of motion for three bodies."""

    positions = state[:9].reshape(3, 3)
    velocities = state[9:].reshape(3, 3)
    accelerations = np.zeros((3, 3))

    for i in range(3):
        for j in range(3):
            if i == j:
                continue

            displacement = positions[j] - positions[i]
            distance = np.linalg.norm(displacement)

            if distance < 1e-10 * LENGTH_SCALE:
                raise ValueError("Gravitational singularity detected.")

            accelerations[i] += (
                G * masses[j] * displacement / distance**3
            )

    return np.concatenate([velocities.ravel(), accelerations.ravel()])


Collision and Ejection Events in 3D

In [ ]:
# ============================================================
# CELL 6 — COLLISION AND EJECTION EVENTS
# ============================================================


# Create a terminal event for one body pair. The event function reaches zero
# when the pair's separation equals collision_radius.
def make_collision_event(i, j):
    """Create a terminal event for a collision between bodies i and j."""

    def collision_event(t, state):
        positions = state[:9].reshape(3, 3)
        distance = np.linalg.norm(positions[i] - positions[j])
        return distance - collision_radius

    collision_event.terminal = True
    collision_event.direction = -1
    return collision_event


# Create a terminal event for body i. It triggers only when the body is both
# beyond ejection_radius from the remaining pair's center of mass and has
# positive specific orbital energy relative to that pair.
def make_ejection_event(i):
    """Create a terminal event for body i becoming unbound and distant."""

    def ejection_event(t, state):
        positions = state[:9].reshape(3, 3)
        velocities = state[9:].reshape(3, 3)

        others = [j for j in range(3) if j != i]
        j, k = others

        pair_mass = masses[j] + masses[k]
        pair_position = (
            masses[j] * positions[j] + masses[k] * positions[k]
        ) / pair_mass
        pair_velocity = (
            masses[j] * velocities[j] + masses[k] * velocities[k]
        ) / pair_mass

        displacement = positions[i] - pair_position
        relative_velocity = velocities[i] - pair_velocity
        distance = np.linalg.norm(displacement)

        specific_energy = (
            0.5 * np.dot(relative_velocity, relative_velocity)
            - G * pair_mass / distance
        )

        distance_condition = distance - ejection_radius
        energy_scale = G * MASS_SCALE / LENGTH_SCALE
        energy_condition = specific_energy / energy_scale

        return min(distance_condition / LENGTH_SCALE, energy_condition)

    ejection_event.terminal = True
    ejection_event.direction = 1
    return ejection_event


# Construct all six possible termination events: three pairwise collisions,
# followed by the possible ejection of each of the three bodies. The ordering
# is also used by run_simulation when identifying the event that stopped a run.
collision_events = [
    make_collision_event(0, 1),
    make_collision_event(0, 2),
    make_collision_event(1, 2),
]
ejection_events = [
    make_ejection_event(0),
    make_ejection_event(1),
    make_ejection_event(2),
]
events = collision_events + ejection_events


Total Mechanical Energy

In [ ]:
# ============================================================
# CELL 7 — TOTAL MECHANICAL ENERGY
# ============================================================

# Calculate the total kinetic energy of all three bodies and add the pairwise
# gravitational potential energies. This quantity is used to monitor numerical
# energy conservation and solver drift.
def calculate_energy(positions, velocities):
    """Calculate total kinetic plus gravitational potential energy."""

    kinetic = 0.5 * np.sum(
        masses[None, :, None] * velocities**2,
        axis=(1, 2)
    )

    potential = np.zeros(len(positions))

    for i in range(3):
        for j in range(i + 1, 3):
            separation = np.linalg.norm(
                positions[:, j] - positions[:, i],
                axis=1
            )
            potential -= G * masses[i] * masses[j] / separation

    return kinetic + potential


Run One Numerical Simulation

In [ ]:
# ============================================================
# CELL 8 — RUN ONE NUMERICAL SIMULATION
# ============================================================

# Integrate one set of initial conditions, stop if a collision or ejection event
# occurs, sample the continuous solution, and calculate energy diagnostics.
# The returned dictionary stores the trajectory, solver metadata, and outcome.
def run_simulation(initial_positions, initial_velocities,
                   rtol=1e-12, atol=1e-14, num_points=5000):
    """Integrate one three-body trajectory and calculate diagnostics."""

    positions0, velocities0 = make_initial_conditions(
        initial_positions, initial_velocities
    )
    initial_state = np.concatenate([positions0.ravel(), velocities0.ravel()])

    solution = solve_ivp(
        derivatives,
        (t_start, t_end),
        initial_state,
        method="DOP853",
        rtol=rtol,
        atol=atol,
        events=events,
        dense_output=True,
        max_step=0.1 * TIME_SCALE
    )

    if not solution.success:
        raise RuntimeError(solution.message)

    event_status = "Undetected collision or ejection"
    event_time = None
    event_names = [
        "Collision: Sun and Earth",
        "Collision: Sun and Moon",
        "Collision: Earth and Moon",
        "Ejection: Sun",
        "Ejection: Earth",
        "Ejection: Moon",
    ]

    for i, event_times in enumerate(solution.t_events):
        if len(event_times) > 0:
            event_status = event_names[i]
            event_time = event_times[0]
            break

    actual_end = solution.t[-1]
    times = np.linspace(t_start, actual_end, num_points)
    sampled_state = solution.sol(times)

    trajectories = sampled_state[:9].T.reshape(-1, 3, 3)
    velocities = sampled_state[9:].T.reshape(-1, 3, 3)

    energy = calculate_energy(trajectories, velocities)
    initial_energy = energy[0]
    energy_scale = G * MASS_SCALE**2 / LENGTH_SCALE

    if abs(initial_energy) < 1e-14 * energy_scale:
        relative_energy_error = np.full_like(energy, np.nan)
        max_energy_error = np.nan
    else:
        relative_energy_error = (energy - initial_energy) / abs(initial_energy)
        max_energy_error = np.max(np.abs(relative_energy_error))

    return {
        "times": times,
        "positions": trajectories,
        "velocities": velocities,
        "energy": energy,
        "relative_energy_error": relative_energy_error,
        "max_energy_error": max_energy_error,
        "initial_energy": initial_energy,
        "function_evaluations": solution.nfev,
        "initial_positions": initial_positions.copy(),
        "initial_velocities": initial_velocities.copy(),
        "rtol": rtol,
        "atol": atol,
        "event_status": event_status,
        "event_time": event_time,
        "actual_end": actual_end,
        "solution": solution,
    }


Baseline Simulation

In [ ]:
# ============================================================
# CELL 9 — BASELINE SIMULATION
# ============================================================

# Run the primary initial-value problem using rtol and atol from Cell 3. The
# resulting trajectory is used for the animation and the primary diagnostics.
baseline = run_simulation(
    initial_positions=initial_positions,
    initial_velocities=initial_velocities,
    rtol=rtol,
    atol=atol,
    num_points=num_frames
)

print("\nBASELINE SIMULATION")
print(f"Initial total energy: {baseline['initial_energy']:.6e} J")
print(f"Maximum relative energy drift: {baseline['max_energy_error']:.6e}")
print(f"Derivative evaluations: {baseline['function_evaluations']}")
print(f"Simulation outcome: {baseline['event_status']}")

if baseline["event_time"] is not None:
    print(f"Event time: {baseline['event_time']:.3f} s")

print(
    f"Actual simulated duration: "
    f"{baseline['actual_end'] / TIME_SCALE:.3f} "
    "characteristic time units"
)


Numerical Convergence Check

In [ ]:
# ============================================================
# CELL 10 — NUMERICAL CONVERGENCE CHECK
# ============================================================

# Repeat the same initial-value problem with progressively tighter solver
# tolerances. First compare maximum relative energy drift, then compare adjacent
# trajectories on a shared time grid to assess sensitivity to solver accuracy.
convergence_runs = []

for rtol_test, atol_test in tolerances:
    result = run_simulation(
        initial_positions=initial_positions,
        initial_velocities=initial_velocities,
        rtol=rtol_test,
        atol=atol_test,
        num_points=5000
    )
    convergence_runs.append(result)

print("\nNUMERICAL CONVERGENCE")

convergence_table = pd.DataFrame([
    {
        "rtol": f"{result['rtol']:.0e}",
        "atol": f"{result['atol']:.0e}",
        "Max relative energy drift": f"{result['max_energy_error']:.6e}",
    }
    for result in convergence_runs
])

display(convergence_table.style.hide(axis="index"))


# Compare neighboring tolerance levels. Interpolate both numerical solutions
# onto the same time grid and calculate maximum and RMS trajectory separation.
print("\nTRAJECTORY DIFFERENCES (rtol vs rtol)")

for coarse, fine in zip(convergence_runs[:-1], convergence_runs[1:]):
    common_end = min(coarse["actual_end"], fine["actual_end"])
    comparison_times = np.linspace(t_start, common_end, 5000)

    coarse_positions = (
        coarse["solution"].sol(comparison_times)
        [:9].T.reshape(-1, 3, 3)
    )
    fine_positions = (
        fine["solution"].sol(comparison_times)
        [:9].T.reshape(-1, 3, 3)
    )

    difference = (coarse_positions - fine_positions) / LENGTH_SCALE
    pointwise_error = np.linalg.norm(difference, axis=2)

    print(
        f"{coarse['rtol']:.0e} vs {fine['rtol']:.0e}: "
        f"Maximum = {np.max(pointwise_error):.6e} L0, "
        f"Global RMS = {np.sqrt(np.mean(pointwise_error**2)):.6e} L0"
    )



Create Simulation Title

In [ ]:
# ============================================================
# CELL 11 — CREATE SIMULATION TITLE
# ============================================================

# Build a descriptive title from the characteristic scales, body masses, and
# dimensionless initial positions and velocities. This title is printed here
# and passed to the animation function in the next cell.
def make_simulation_title(initial_positions, initial_velocities):
    """Format masses, dimensionless initial conditions, and SI scales."""

    title_lines = [
        "Sun-Earth-Moon Gravitational Simulation",
        f"Scales: L₀ = {LENGTH_SCALE} m  |  "
        f"v₀ = {VELOCITY_SCALE} m/s",
    ]

    for i in range(3):
        x, y, z = initial_positions[i]
        vx, vy, vz = initial_velocities[i]
        title_lines.append(
            f"{['Sun', 'Earth', 'Moon'][i]}: m = {masses[i]:.6e} kg  |  "
            f"r = ({x}, {y}, {z}) × L₀  |  "
            f"v = ({vx}, {vy}, {vz}) × v₀"
        )

    return "\n".join(title_lines)


simulation_title = make_simulation_title(
    initial_positions,
    initial_velocities
)

print("\nSIMULATION TITLE")
print(simulation_title)


Create 3D Animation

In [ ]:

# ============================================================
# CELL 12 — CREATE 3D ANIMATION
# ============================================================

# Calculate the center of mass at every sampled time. The camera uses this
# point as its moving origin, so the animation follows the barycenter of
# the Sun-Earth-Moon system.
def calculate_center_of_mass(positions):
    """Return the center of mass at each time sample."""

    center_of_mass = np.sum(
        positions * masses[None, :, None],
        axis=1
    ) / np.sum(masses)

    return center_of_mass


# Determine the displayed axis scale from LENGTH_SCALE.
# The axis unit is automatically set one order of magnitude below L₀.
axis_exponent = int(np.floor(np.log10(LENGTH_SCALE)) - 1)
axis_scale = 10.0 ** axis_exponent


# Create a 3D animation from the baseline trajectory. Positions are recentered
# on the center of mass and displayed using the automatically selected axis
# scale. The animation includes moving body markers, trajectory trails,
# a simulation clock, and title.
def create_animation(baseline, simulation_title):
    """Create the center-of-mass-following 3D trajectory animation."""

    times = baseline["times"]
    positions = baseline["positions"]

    # Calculate the center of mass and recenter the trajectories.
    center_of_mass = calculate_center_of_mass(positions)
    positions_relative = positions - center_of_mass[:, None, :]

    # Convert physical positions into the automatically selected axis units.
    positions_display = positions_relative / axis_scale

    # Determine the plot boundaries from the physical length scale
    # and the maximum extent of the simulated trajectories.
    maximum_extent = max(
        3 * LENGTH_SCALE / axis_scale,
        1.15 * np.max(np.abs(positions_display))
    )

    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")

    ax.set_xlim(-maximum_extent, maximum_extent)
    ax.set_ylim(-maximum_extent, maximum_extent)
    ax.set_zlim(-maximum_extent, maximum_extent)
    ax.set_box_aspect((1, 1, 1))

    # Automatically update axis labels to match the selected scale.
    ax.set_xlabel(rf"X ($10^{{{axis_exponent}}}$ m)")
    ax.set_ylabel(rf"Y ($10^{{{axis_exponent}}}$ m)")
    ax.set_zlabel(rf"Z ($10^{{{axis_exponent}}}$ m)")

    fig.suptitle(simulation_title, fontsize=8, y=0.99)
    fig.subplots_adjust(top=0.82)
    ax.grid(True)

    colors = ["red", "blue", "green"]
    body_names = ["Sun", "Earth", "Moon"]

    body_markers = []
    body_trails = []

    for i in range(3):
        marker, = ax.plot(
            [], [], [], "o", color=colors[i], markersize=8,
            label=body_names[i]
        )
        trail, = ax.plot(
            [], [], [], color=colors[i], linewidth=1, alpha=0.6
        )
        body_markers.append(marker)
        body_trails.append(trail)

    ax.legend(loc="upper right")
    time_text = fig.text(0.02, 0.02, "")

    def initialize():
        for marker, trail in zip(body_markers, body_trails):
            marker.set_data([], [])
            marker.set_3d_properties([])
            trail.set_data([], [])
            trail.set_3d_properties([])
        time_text.set_text("")
        return body_markers + body_trails + [time_text]

    def update(frame):
        for i in range(3):
            x = positions_display[:frame + 1, i, 0]
            y = positions_display[:frame + 1, i, 1]
            z = positions_display[:frame + 1, i, 2]

            body_markers[i].set_data([x[-1]], [y[-1]])
            body_markers[i].set_3d_properties([z[-1]])

            body_trails[i].set_data(x, y)
            body_trails[i].set_3d_properties(z)

        current_time = times[frame]
        time_text.set_text(
            f"Time: {current_time / TIME_SCALE:.2f} characteristic units"
        )
        return body_markers + body_trails + [time_text]

    animation = FuncAnimation(
        fig,
        update,
        frames=len(times),
        init_func=initialize,
        interval=1000 / fps,
        blit=False
    )

    plt.close(fig)
    return animation


animation = create_animation(baseline, simulation_title)
print("Sun-Earth-Moon animation created successfully.")


Export and Display Video

In [ ]:
# ============================================================
# CELL 13 — EXPORT AND DISPLAY VIDEO
# ============================================================

# Choose the MP4 filename. The video is saved in the notebook's current
# working directory. fps is defined in Cell 3; bitrate and dpi control the
# encoded video quality and rendered resolution.
video_filename = "3BP_sun_earth_moon.mp4"

writer = FFMpegWriter(
    fps=fps,
    bitrate=1800
)

# Export the animation object created in Cell 12.
animation.save(
    video_filename,
    writer=writer,
    dpi=150
)

print(f"Video saved: {video_filename}")

# Embed the exported animation in the notebook output for preview.
display(HTML(animation.to_html5_video()))

print("\nVIDEO EXPORT COMPLETE")
print(f"Saved: {video_filename}")
print(f"Simulation outcome: {baseline['event_status']}")
print(
    f"Maximum relative energy drift: "
    f"{baseline['max_energy_error']:.6e}"
)
